In [1]:
import os
from datetime import datetime

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

# Get MLflow configuration from environment variables
MLFLOW_TRACKING_URI= ""
MLFLOW_EXPERIMENT_NAME= ""
MLFLOW_TRACKING_PASSWORD= ""
MLFLOW_TRACKING_USERNAME= ""
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD
os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
print(f"MLflow Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment Name: {MLFLOW_EXPERIMENT_NAME}") 
print(f"Password: {MLFLOW_TRACKING_PASSWORD}")

MLflow Tracking URI: 
Experiment Name: 
Password: 


In [2]:
# Set up MLflow client
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

# Get experiment by name
try:
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    print(f"Experiment: {experiment}")
    if experiment is None:
        raise ValueError(f"Experiment '{MLFLOW_EXPERIMENT_NAME}' not found")

    experiment_id = experiment.experiment_id
    print(f"Found experiment: {experiment.name} (ID: {experiment_id})")
except Exception as e:
    print(f"Error getting experiment: {e}")
    experiment_id = None

Experiment: None
Error getting experiment: Experiment '' not found


/opt/anaconda3/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [3]:
# Search for traces in the experiment
trace_data = []
if experiment_id:
    try:
        traces = client.search_traces(
            locations=[experiment_id],
            max_results=100,
            order_by=["timestamp_ms DESC"]
        )
        # Convert traces to a more readable format
        import json
        for trace in traces:                      

            request_obj = None
            if isinstance(trace.data.request, str):
                request_obj = json.loads(trace.data.request)

            response_obj = trace.data.response
                                                                                                                                             
            # If it's a string, parse it                                                                                                                  
            if isinstance(response_obj, str):
                response_obj = json.loads(response_obj) 

            # Get the trace breakdown (spans)
            spans = trace.data.spans

            span_names = [span.name for span in trace.data.spans]
                    # Now get the actual response text                                                                                                            
            response_text = response_obj['response']
            trace_info = {
                "request": request_obj,
                "response": response_text,
                "trace_id": trace.info.request_id,
                "timestamp": datetime.fromtimestamp(trace.info.timestamp_ms / 1000),
                "status": trace.info.status,
                "execution_time_ms": trace.info.execution_time_ms,
                "tags": trace.info.tags if hasattr(trace.info, 'tags') else {}
            }
            trace_data.append(trace_info)

        # Display as DataFrame
        if trace_data:
            df_traces = pd.DataFrame(trace_data)
            display(df_traces)
        else:
            print("No traces found in this experiment")

    except Exception as e:
        print(f"Error searching traces: {e}")
        traces = []
else:
    print("Cannot search traces without valid experiment_id")

Cannot search traces without valid experiment_id


# Analysis IDEA list

# hallicinuation check (that we dont want)
# security concerns like leaked sensitive data
# cost analysis on token usuage
# reasoning drift


In [4]:
for obj in trace_data:
    for key,value in obj.items():
        print(f"{key}: {value}")